# FISH Coordinate Imputation Tutorial

This tutorial demonstrates how to impute missing 3D coordinates in multiplexed DNA FISH data using uchrom's `im.impute` module.

## Overview

Multiplexed FISH experiments often have ~20-40% missing coordinates due to:
- Weak signals
- Probe overlap
- Optical issues

We provide three imputation methods:
1. **Linear interpolation** - Fast, no extra dependencies
2. **Cubic spline** - Smooth, no extra dependencies
3. **SnapFISH-IMPUTE** - Similarity-weighted, best accuracy (requires scikit-image)

In [ ]:
import uchrom as uc
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Load Test Data

We'll use a small subset of mESC Sox2 data (Huang et al. 2021) with 5 cells and ~30% missing coordinates.

In [ ]:
# Load data with missing coordinates
cd = uc.ChromData.read('example-data/mesc_sox2/mESC_Sox2_5cells_wnan.h5cd')

print(f"Data shape: {cd.coords.shape}")
print(f"Cells: {cd.spots['trace_id'].nunique()}")
print(f"Loci per cell: {len(cd.spots) // cd.spots['trace_id'].nunique()}")

n_missing = np.isnan(cd.coords).sum()
n_total = cd.coords.size
print(f"\nMissing coordinates: {n_missing}/{n_total} ({n_missing/n_total*100:.1f}%)")

## 2. Visualize Missing Data Pattern

In [ ]:
# Count missing coordinates per trace
import pandas as pd

missing_per_trace = []
for trace_id, group_df in cd.spots.groupby('trace_id'):
    indices = group_df.index
    n_missing_trace = np.isnan(cd.coords[indices]).any(axis=1).sum()
    missing_per_trace.append({
        'trace_id': trace_id,
        'n_missing': n_missing_trace,
        'n_total': len(indices)
    })

df_missing = pd.DataFrame(missing_per_trace)
df_missing['missing_pct'] = df_missing['n_missing'] / df_missing['n_total'] * 100

plt.figure(figsize=(10, 4))
plt.bar(range(len(df_missing)), df_missing['missing_pct'])
plt.axhline(y=df_missing['missing_pct'].mean(), color='r', linestyle='--',
            label=f'Mean: {df_missing["missing_pct"].mean():.1f}%')
plt.xlabel('Cell')
plt.ylabel('Missing %')
plt.title('Missing Coordinate Rate per Cell')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(df_missing[['trace_id', 'n_missing', 'n_total', 'missing_pct']])

## 3. Method 1: Linear Interpolation

Fast and simple - linearly interpolates along the genomic coordinate.

In [ ]:
from uchrom.im.impute import impute_coordinates

cd_linear = impute_coordinates(cd, method='linear')

n_missing_after = np.isnan(cd_linear.coords).sum()
n_imputed = n_missing - n_missing_after

print(f"Before: {n_missing} missing coordinates")
print(f"After:  {n_missing_after} missing coordinates")
print(f"Imputed: {n_imputed} ({n_imputed/n_missing*100:.1f}% of missing)")

## 4. Method 2: Cubic Spline

Smoother than linear - uses cubic spline interpolation.

In [ ]:
cd_cubic = impute_coordinates(cd, method='cubic')

n_missing_cubic = np.isnan(cd_cubic.coords).sum()
print(f"Cubic imputation: {n_missing - n_missing_cubic} coordinates imputed")

## 5. Method 3: SnapFISH-IMPUTE Algorithm

Most accurate - uses similarity weighting across cells. Requires `scikit-image`
(`pip install u-chrom[snapfish]`).

**Note**: This cell will be skipped if the dependency is not installed.

In [ ]:
try:
    cd_snapfish = impute_coordinates(
        cd,
        method='snapfish',
        max_iter=10,
        n_jobs=1,        # set >1 to parallelize the L-BFGS recovery across CPU cores
        verbose=True,
    )

    n_missing_snapfish = np.isnan(cd_snapfish.coords).sum()
    print(f"\nSnapFISH imputed: {n_missing - n_missing_snapfish} coordinates")
    print("(Any still-NaN loci had no usable reference data.)")

except ImportError as e:
    print(f"⚠ SnapFISH requires an extra dependency: {e}")
    print("Install with: pip install u-chrom[snapfish]   # adds scikit-image")
    cd_snapfish = None

## 5b. Handling Absent Rows (truly missing loci)

In real FISH / chromatin-tracing data, a missing locus is often an **absent row** —
the locus simply has no entry for that trace — rather than a row with NaN coordinates.

All imputation methods here first **densify** the data onto the per-chromosome locus
panel (the union of observed loci, or an explicit `ann` panel), inserting NaN rows for
absent loci, then fill them. The output is therefore the full dense grid, with NaN only
where no reference data was available. This mirrors SnapFISH-IMPUTE's `fill_missing_by_nan`
preprocessing step.

In [ ]:
# Simulate real data where missing loci are ABSENT rows (not NaN rows):
# drop every NaN-coordinate row so those loci simply have no entry.
keep = ~np.isnan(cd.coords[:, 0])
cd_absent = uc.ChromData(
    coords=cd.coords[keep],
    spots=cd.spots[keep].reset_index(drop=True),
)
print(f"Absent-row input : {len(cd_absent.spots)} rows, "
      f"{int(np.isnan(cd_absent.coords).sum())} NaN coords (none — loci are simply absent)")

# Imputation densifies back to the full panel and fills the gaps.
cd_absent_filled = impute_coordinates(cd_absent, method='linear')
print(f"After imputation : {len(cd_absent_filled.spots)} rows "
      f"(densified back to the full locus panel)")
print(f"NaN remaining    : {int(np.isnan(cd_absent_filled.coords).sum())}")

# Tip: pass an explicit `ann` panel to also restore loci absent from EVERY trace, e.g.
#   ann = pd.read_csv('example-data/mesc_sox2/mESC_Sox2_ann_chr129.txt', sep='\t')  # region/pos/start/end
#   impute_coordinates(cd_absent, method='linear', ann=ann)

## 6. Compare Methods

Visualize one cell before and after imputation.

In [ ]:
# Pick the first cell/trace
trace_id = cd_linear.spots['trace_id'].unique()[0]

# Align by genomic locus (chrom, start), NOT by row position: imputation returns
# a NEW ChromData on the canonical dense grid, whose row order may differ from the
# input. Keying by 'start' is robust to reordering and to densified (absent) rows.
def coords_by_start(cd_x, trace_id):
    sub = cd_x.spots[cd_x.spots['trace_id'] == trace_id]
    return {int(s): cd_x.coords[i] for i, s in zip(sub.index, sub['start'])}

panel = sorted(int(s) for s in
               cd_linear.spots.loc[cd_linear.spots['trace_id'] == trace_id, 'start'])
orig_map, lin_map, cub_map = (coords_by_start(c, trace_id)
                              for c in (cd, cd_linear, cd_cubic))

coords_orig = np.array([orig_map.get(s, [np.nan] * 3) for s in panel])
coords_linear = np.array([lin_map[s] for s in panel])
coords_cubic = np.array([cub_map[s] for s in panel])
missing_mask = np.isnan(coords_orig[:, 0])

fig = plt.figure(figsize=(15, 4))

ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(coords_orig[~missing_mask, 0], coords_orig[~missing_mask, 1],
            coords_orig[~missing_mask, 2], c='blue', s=20, label='Original')
ax1.set_title(f'Original\n({missing_mask.sum()} missing)')
ax1.legend()

ax2 = fig.add_subplot(132, projection='3d')
ax2.scatter(coords_linear[~missing_mask, 0], coords_linear[~missing_mask, 1],
            coords_linear[~missing_mask, 2], c='blue', s=20, label='Original', alpha=0.5)
ax2.scatter(coords_linear[missing_mask, 0], coords_linear[missing_mask, 1],
            coords_linear[missing_mask, 2], c='red', s=20, label='Imputed')
ax2.set_title('Linear Interpolation')
ax2.legend()

ax3 = fig.add_subplot(133, projection='3d')
ax3.scatter(coords_cubic[~missing_mask, 0], coords_cubic[~missing_mask, 1],
            coords_cubic[~missing_mask, 2], c='blue', s=20, label='Original', alpha=0.5)
ax3.scatter(coords_cubic[missing_mask, 0], coords_cubic[missing_mask, 1],
            coords_cubic[missing_mask, 2], c='red', s=20, label='Imputed')
ax3.set_title('Cubic Spline')
ax3.legend()

plt.tight_layout()
plt.show()

## 7. Save Imputed Data

In [ ]:
# Save the imputed data
cd_linear.write('example-data/mesc_sox2/mESC_Sox2_5cells_imputed_linear.h5cd')
print("✓ Saved: example-data/mesc_sox2/mESC_Sox2_5cells_imputed_linear.h5cd")

if cd_snapfish is not None:
    cd_snapfish.write('example-data/mesc_sox2/mESC_Sox2_5cells_imputed_snapfish.h5cd')
    print("✓ Saved: example-data/mesc_sox2/mESC_Sox2_5cells_imputed_snapfish.h5cd")

## 8. Downstream Analysis

Now you can use the imputed data for downstream analysis like TAD calling, compartment detection, etc.

In [ ]:
# Example: Continue with TAD calling
# from uchrom.strc.tad import get_domains
# ...

print("Imputed ChromData is ready for:")
print("- TAD detection (uchrom.strc.tad)")
print("- Loop calling (uchrom.strc.loop)")
print("- Compartment analysis (uchrom.strc.comp)")
print("- Visualization (uchrom.browser)")

## Summary

### Key Takeaways

1. **Linear/Cubic** - Fast, no dependencies, good for small gaps
2. **SnapFISH** - Most accurate, uses cell similarity, needs scikit-image
3. **All methods** preserve original non-missing coordinates
4. **Absent rows are handled** - methods densify onto the locus panel, so loci
   missing as absent rows (not just NaN rows) are imputed too; pass `ann=` to
   also restore loci absent from every trace
5. **SnapFISH scales** - pass `n_jobs>1` to parallelize the recovery step across cores

### When to Use Which Method

- **Linear**: Quick exploratory analysis, <20% missing
- **Cubic**: Smoother than linear, <20% missing
- **SnapFISH**: Best accuracy, >20% missing, have multiple cells

### Performance

| Method | Time (5 cells) | Dependencies |
|--------|----------------|-------------|
| Linear | <1 sec | scipy (core) |
| Cubic | <1 sec | scipy (core) |
| SnapFISH | ~10 sec (n_jobs=1) | scikit-image (`pip install u-chrom[snapfish]`) |

For large datasets, `impute_coordinates(cd, method='snapfish', n_jobs=8)` gives
roughly a 4x speedup (process-based; the per-haploid L-BFGS recovery dominates runtime).

### References

- **SnapFISH-IMPUTE**: Yu et al. (2023) https://github.com/hyuyu104/SnapFISH-IMPUTE
- **Test data**: Huang et al. (2021) *Nature Genetics* doi:10.1038/s41588-021-00863-6